# <font color='green'>SINTETIZADOR</font>
### <font color='green'>COPULA GAUSSIAN</font>

In [1]:
# Validar ambiente de execução do Python
import sys
#print(sys.executable)

### 1. IMPORTAR E CONFIGURAR

In [2]:
# Importações e configuração

# Biblioteca padrão
import os
from pathlib import Path
from platform import python_version

# Manipulação e visualização de dados
import pandas as pd

# Estatística
from scipy.stats import normaltest

# Geração e avaliação de dados sintéticos
from sdv.evaluation.single_table import (
    evaluate_quality,
    get_column_plot,
    run_diagnostic,
)
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

# Versão Python 
print('Python:',python_version())

Python: 3.10.13


> **Nota sobre o kernel:** se uma importação do SDV/PyTorch for interrompida e surgir o erro `Only a single TORCH_LIBRARY ... triton`, reinicie o kernel antes de executar novamente. O PyTorch mantém registros nativos na memória que não podem ser corrigidos apenas repetindo o `import`.

In [3]:
# Exibir as versões das bibliotecas
%reload_ext watermark
%watermark --iversions

pandas  : 2.3.3
platform: 1.0.8
scipy   : 1.15.3
sdv     : 1.37.0



### 2. AMBIENTE DE EXECUÇÃO

In [4]:
# Configurar o diretório do projeto
project_root = Path.cwd()

if project_root.name == "notebooks":
    os.chdir(project_root.parent)

#print(f"Diretório do projeto: {Path.cwd().name}")

### 3. CARREGAR DADOS

In [5]:
# Carregar dados originais
date_columns = ["creation_date", "last_activity_date"]

df_origin = pd.read_csv("data/processed/customers.csv",)

df_origin[date_columns] = df_origin[date_columns].apply(
    pd.to_datetime,
    format="%m-%d-%Y %H:%M:%S",
    errors="raise",
)

In [6]:
# Copiar dados originais  
df = df_origin.copy()

### 4. INSPECIONAR DADOS
> Responde como o conjunto dos dados está estruturado?

In [7]:
# Primeiros registros 
df.head()

,firstname,lastname,email,address,country,last_country_logged,creation_date,last_activity_date,age_group,id
0,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,2023-02-17,2023-03-02 00:43:50,4.0,280a5c38-422b-4f83-b472-5bac6450b11a
1,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,2022-04-23,2023-03-02 00:43:50,1.0,07de7b22-b1a1-4fa0-870e-67fbafc3b347
2,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,2021-05-28,2023-03-02 00:43:50,10.0,f97b472d-b859-450e-b053-87824e3ad5e0
3,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,2021-08-26,2023-03-02 00:43:50,3.0,c892ddd3-54d0-42cc-a784-98196241ff60
4,Aaron,Smith,megankirby@anderson.com,"5223 Stacie Lodge Suite 258\nSouth Nathanton, ...",TLS,TLS,2022-11-19,2023-03-02 00:43:50,8.0,d81a15a5-2d1b-41d0-b0c3-0f5ff3322c49


In [8]:
## Informações do DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99244 entries, 0 to 99243
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   firstname            99244 non-null  object        
 1   lastname             99244 non-null  object        
 2   email                99244 non-null  object        
 3   address              99244 non-null  object        
 4   country              99244 non-null  object        
 5   last_country_logged  99244 non-null  object        
 6   creation_date        99244 non-null  datetime64[ns]
 7   last_activity_date   99244 non-null  datetime64[ns]
 8   age_group            99244 non-null  float64       
 9   id                   99244 non-null  object        
dtypes: datetime64[ns](2), float64(1), object(7)
memory usage: 7.6+ MB


### 5. CRIAR E CONFIGURAR OS METADADOS

In [9]:
# Definir nomes e diretório de saída
table_name = "customers"
output_directory = Path("data/synthetic")

# Criar os metadados a partir do DataFrame
metadata = Metadata.detect_from_dataframe(
    data=df,
    table_name=table_name,
)

# Exibir os metadados como DataFrame
pd.DataFrame(metadata.to_dict()["tables"][table_name]["columns"]).T

,pii,sdtype
firstname,True,first_name
lastname,True,last_name
email,True,email
address,NaN,categorical
country,NaN,categorical
last_country_logged,NaN,categorical
creation_date,NaN,datetime
last_activity_date,NaN,datetime
age_group,NaN,numerical
id,NaN,id


`Nota técnica`

> As colunas country e last_country_logged foram mantidas como categóricas, pois armazenam códigos de países no padrão ISO Alpha-3 (três letras), enquanto o tipo country_code do SDV é destinado a códigos ISO Alpha-2 (duas letras).


#### 5.1. SALVAR OS METADADOS

In [10]:
# Salvar os metadados
metadata_file = output_directory / f"{table_name}_metadata.json"

output_directory.mkdir(parents=True, exist_ok=True)
metadata.save_to_json(metadata_file, mode="overwrite")

print(f"Metadados salvos em: {metadata_file}")

Metadados salvos em: data/synthetic/customers_metadata.json


### 6. CRIAR DADOS SINTÉTICOS

In [11]:
# Criar sintetizador
synthesizer = GaussianCopulaSynthesizer(metadata)

In [12]:
# Treinar o sintetizador com os dados reais
synthesizer.fit(df)   

In [13]:
# Definir a quantidade de registros sintéticos a serem gerados
num_rows = len(df)  # Mesmo tamanho do conjunto original

# Reinicia o estado inicial de geração dos dados sintéticos.
synthesizer.reset_sampling()

# Gerar os dados sintéticos
synthetic_df = synthesizer.sample(num_rows=num_rows)

In [14]:
# Preservar a chave primária para manter os relacionamentos entre tabelas
primary_key = "id"
synthetic_df[primary_key] = df[primary_key].to_numpy(copy=True)

> A chave primária original é preservada para manter compatibilidade com relacionamentos entre tabelas. 

In [15]:
# Confirmar valores, ordem, nome e tipo da coluna usada nos relacionamentos.
pd.testing.assert_series_equal(
    synthetic_df[primary_key],
    df[primary_key],
    check_dtype=True,
    check_names=True
)

print(f"Validação concluída: a coluna '{primary_key}' foi preservada.")

Validação concluída: a coluna 'id' foi preservada.


In [16]:
# Manter a mesma ordem de colunas do conjunto original
synthetic_df = synthetic_df.loc[:, df.columns]

### 7. INSPECIONAR DADOS SINTÉTICOS
> Responde como o conjunto dos dados está estruturado?

In [17]:
# Visualizar os primeiros dados sintéticos
synthetic_df.head()

,firstname,lastname,email,address,country,last_country_logged,creation_date,last_activity_date,age_group,id
0,Chelsea,Hill,robertsonjason@example.org,66350 Samantha Rue Suite 184\nSouth Melaniefor...,DMA,DMA,2022-10-10,2023-03-02 07:16:47,2.0,280a5c38-422b-4f83-b472-5bac6450b11a
1,Vincent,Foster,xguzman@example.net,"5427 Dennis Fields\nClaudiaview, NC 85882",NOR,KEN,2022-10-14,2023-03-01 17:18:22,6.0,07de7b22-b1a1-4fa0-870e-67fbafc3b347
2,Maria,Alexander,suttontheresa@example.net,"15557 Timothy Rapids Apt. 094\nSouth Michelle,...",TKM,TKM,2022-07-21,2023-03-01 12:55:57,5.0,f97b472d-b859-450e-b053-87824e3ad5e0
3,Bradley,Johnson,monica38@example.org,"95642 Elizabeth Mountains\nOwensside, CA 99006",MLI,LBY,2022-02-08,2023-03-01 12:53:23,5.0,c892ddd3-54d0-42cc-a784-98196241ff60
4,Jennifer,Clayton,sburton@example.net,"01315 Patterson Squares Suite 659\nThomasstad,...",PNG,PNG,2022-10-21,2023-03-01 07:45:12,6.0,d81a15a5-2d1b-41d0-b0c3-0f5ff3322c49


In [18]:
# Verificar dimensões dos dados sintéticos
synthetic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99244 entries, 0 to 99243
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   firstname            99244 non-null  object        
 1   lastname             99244 non-null  object        
 2   email                99244 non-null  object        
 3   address              99244 non-null  object        
 4   country              99244 non-null  object        
 5   last_country_logged  99244 non-null  object        
 6   creation_date        99244 non-null  datetime64[ns]
 7   last_activity_date   99244 non-null  datetime64[ns]
 8   age_group            99244 non-null  float64       
 9   id                   99244 non-null  object        
dtypes: datetime64[ns](2), float64(1), object(7)
memory usage: 7.6+ MB


In [19]:
# Total de valores nulos por coluna
synthetic_df.isnull().sum()

firstname              0
lastname               0
email                  0
address                0
country                0
last_country_logged    0
creation_date          0
last_activity_date     0
age_group              0
id                     0
dtype: int64

In [20]:
# Verificar a quantidade de registros duplicados
int(synthetic_df.duplicated().sum())    

0

In [21]:
# Total de ids duplicados
int(synthetic_df.duplicated(subset="id").sum())

0

### 8. ANÁLISE EXPLORATÓRIA DADOS SINTÉTICOS - EDA
> Responde quais padrões, distribuições e características estatísticas existem nos dados?

In [22]:
# Estatísticas específicas das variáveis numéricas
synthetic_df.describe(include="number")

,age_group
count,99244.000000
mean,4.102606
std,3.056627
min,0.000000
25%,1.000000
50%,4.000000
75%,7.000000
max,10.000000


`Nota Técnica`

> Em comparação com a estatística descritiva do arquivo `01_data_analysis`, o conjunto sintético manteve o mesmo volume de registros válidos para `age_group` (99.244), sem valores ausentes, e preservou a amplitude da variável, com mínimo 0 e máximo 10. No entanto, houve deslocamento da distribuição para grupos etários mais baixos: a média caiu de 5,00 nos dados reais para 4,10 nos dados sintéticos, e a mediana passou de 5 para 4. O primeiro quartil também se deslocou de 3 para 1, enquanto o terceiro quartil permaneceu em 7, indicando maior concentração sintética nos grupos inferiores. O desvio-padrão aumentou levemente de 2,92 para 3,06, sugerindo dispersão um pouco maior. Assim, embora a estrutura discreta e o domínio de `age_group` tenham sido preservados, a distribuição sintética não reproduz integralmente a centralidade observada no conjunto original.

In [23]:
# Teste de normalidade para a variável age_group (Teste D'Agostino-Pearson)
variavel = "age_group"
dados = synthetic_df[variavel].dropna()

stat, p_value = normaltest(dados)

print(f"Teste D'Agostino-Pearson para {variavel}")
print(f"Estatística: {stat:.4f}")
print(f"p-valor: {p_value:.4f}")

if p_value > 0.05:
    print("Não há evidência suficiente para rejeitar normalidade.")
else:
    print("Há evidência para rejeitar normalidade.")

Teste D'Agostino-Pearson para age_group
Estatística: 35057.5277
p-valor: 0.0000
Há evidência para rejeitar normalidade.


`Nota Técnica`

> Assim como no arquivo `01_data_analysis`, o teste de normalidade de D'Agostino-Pearson aplicado à variável `age_group` resultou em p-valor 0,0000, levando à rejeição da hipótese de normalidade. 

### 9. AVALIAR OS DADOS SINTÉTICOS

#### 9.1. Run Diagnostic
Verifica se os dados sintéticos estão consistentes com os dados reais e com a estrutura definida nos metadados gerados a partir do conjunto original.

In [24]:
# Validar a estrutura e a consistência dos dados sintéticos
diagnostic = run_diagnostic(
    real_data=df,
    synthetic_data=synthetic_df,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 10/10 [00:00<00:00, 108.69it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 321.85it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



> `Data Validity:` verifica se os valores sintéticos respeitam os tipos, formatos, domínios e restrições definidos nos metadados.

> `Data Structure:` verifica se a estrutura do conjunto sintético (colunas, tipos semânticos e relacionamentos definidos nos metadados) está consistente com a dos dados reais.

#### 9.2. Evaluate Quality
Avalia o quanto os dados sintéticos preservam as características estatísticas dos dados reais.

In [25]:
# Avaliar a semelhança entre os dados reais e sintéticos
quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_df,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 10/10 [00:00<00:00, 43.60it/s]|
Column Shapes Score: 95.07%

(2/2) Evaluating Column Pair Trends: |██████████| 45/45 [00:00<00:00, 101.61it/s]|
Column Pair Trends Score: 17.85%

Overall Score (Average): 56.46%



`Column Shapes:` avalia o quanto a distribuição de cada coluna sintética é semelhante à distribuição da respectiva coluna nos dados reais.

`Column Pair Trends:` avalia o quanto os relacionamentos estatísticos entre pares de colunas foram preservados nos dados sintéticos.

`Nota técnica:` o diagnóstico apresentou 100% de validade estrutural, mas o relatório de qualidade ficou em 56,5%. A diferença ocorre porque o conjunto sintético respeita tipos, formatos e estrutura esperados, porém preserva com baixa fidelidade os relacionamentos estatísticos entre pares de colunas, especialmente em `Column Pair Trends`, como demonstrado mais detalhadamente a seguir.

In [26]:
# Detalhar a qualidade da distribuição de cada coluna
quality_report.get_details("Column Shapes")

,Column,Metric,Score
0,address,TVComplement,0.969872
1,country,TVComplement,0.983052
2,last_country_logged,TVComplement,0.982820
3,creation_date,KSComplement,0.939875
4,last_activity_date,KSComplement,0.964149
5,age_group,KSComplement,0.864274


In [27]:
# Detalhar a qualidade dos relacionamentos estatísticos entre pares de colunas
quality_report.get_details("Column Pair Trends")

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?
0,address,country,ContingencySimilarity,0.00806,NaN,NaN,1.000000,True
1,address,last_country_logged,ContingencySimilarity,0.00780,NaN,NaN,1.000000,True
2,address,creation_date,ContingencySimilarity,NaN,NaN,NaN,0.251615,False
3,address,last_activity_date,ContingencySimilarity,0.10184,NaN,NaN,1.000000,True
4,address,age_group,ContingencySimilarity,NaN,NaN,NaN,0.114499,False
5,country,last_country_logged,ContingencySimilarity,0.28590,NaN,NaN,1.000000,True
6,country,creation_date,ContingencySimilarity,NaN,NaN,NaN,0.140858,False
7,country,last_activity_date,ContingencySimilarity,0.32928,NaN,NaN,0.564563,True
8,country,age_group,ContingencySimilarity,NaN,NaN,NaN,0.060670,False
9,last_country_logged,creation_date,ContingencySimilarity,NaN,NaN,NaN,0.139442,False


#### 9.3. VISUALIZAR AS DISTRIBUIÇÕES

Os gráficos abaixo comparam os dados reais e sintéticos por duas perspectivas complementares.

##### Frequência de cada grupo de idade

In [28]:
# Comparar os grupos de idade
fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_df,
    column_name="age_group",
    metadata=metadata,
    plot_type="bar"
)

fig.show(renderer="plotly_mimetype")

##### Distribuição dos grupos de idade

In [29]:
fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_df,
    column_name="age_group",
    metadata=metadata,
    plot_type="distplot",
)

fig.show(renderer="plotly_mimetype")

### 10. EXPORTAR DADOS SINTÉTICOS

In [30]:
# Persistir o conjunto de dados sintético em CSV com formatação de datas
synthetic_df.to_csv(
    "data/synthetic/customers_synthetic.csv",
    index=False,
    date_format="%m-%d-%Y %H:%M:%S",)

### 11. CONCLUSÃO

O `GaussianCopulaSynthesizer` gerou um conjunto sintético com a mesma estrutura do conjunto original e sem violações de validade detectadas pelo diagnóstico do SDV. As distribuições individuais foram preservadas com boa qualidade geral, mas os relacionamentos estatísticos entre pares de colunas apresentaram baixa fidelidade nesta execução. Portanto, este notebook deve ser interpretado como uma prova de conceito reprodutível para geração e avaliação inicial de dados sintéticos, não como uma solução final de anonimização ou substituição analítica dos dados reais.
